In [ ]:
import gymnasium as gym
import numpy as np

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
master_df = pd.read_csv("./trajectories_db_analysis/master_df.csv")
master_df = pd.get_dummies(
    master_df,
    columns=["original_category"],
    drop_first=False,
    dtype=int
)
master_df.head()

In [ ]:
from sklearn.model_selection import train_test_split

circuits = master_df["original_circuit"].unique()

train_circuits, test_circuits = train_test_split(
    circuits,
    test_size=0.2,
    random_state=42
)

train_df = master_df[master_df["original_circuit"].isin(train_circuits)].copy()
test_df = master_df[master_df["original_circuit"].isin(test_circuits)].copy()

In [ ]:
class OptimizerEnv(gym.Env):

    def __init__(self, df, feature_columns):
        super().__init__()

        # Dataframe with all circuit features and output values
        self.df = df

        # List of feature columns to be used as input for the agent
        self.feature_columns = feature_columns
        
        # Define the observation space as a the number of features in the dataframe
        self.observation_space = gym.spaces.Box(
            low=0,
            high=np.inf,
            shape=(len(feature_columns),),
            dtype=np.float32
        )

        # Define the action space as a discrete space with 5 possible actions (different optimization strategies)
        self.action_space = gym.spaces.Discrete(5)

        # Number of optimization steps taken
        self.step_count = 0

        # Current state of the environment (feature values of the selected row)
        self.state = None

        self._action_to_optimizer = {
            0: "qiskit_standard",
            1: "qiskit_ai",
            2: "tket",
            3: "wisq_rules",
            4: "wisq_bqskit"
        }
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0

        # Selects random row in dataframe
        idx = self.np_random.integers(0, len(self.df))
        self.current_row = self.df.iloc[idx]

        # Sets the state to the feature values of the selected row
        self.state = (
            self.current_row[self.feature_columns].values.astype(np.float32)
        )

        return self.state, {}
    
    def step(self, action):
        action = int(action)

        optimizer = self._action_to_optimizer[action]
        self.step_count += 1

        mask = (
            (self.df["original_circuit"] == self.current_row["original_circuit"]) &
            (self.df["opt_1"] == optimizer)
        )


        matches = self.df.loc[mask]

        if matches.empty:
            raise ValueError(
                f"No row found for circuit={self.current_row['original_circuit']} "
                f"and optimizer={optimizer}"
            )

        optimized_row = matches.iloc[0][self.feature_columns].to_numpy(dtype=np.float32)
        self.state = optimized_row

        old_2q = matches.iloc[0]["initial_two_qubit_gates"]
        new_2q = matches.iloc[0]["final_two_qubit_gates"]
        
        reward = 0
        if old_2q == 0:
            reward = 0.0
        else:
            reward = (old_2q - new_2q) / old_2q
        
        terminated = True

        truncated = False

        info = {"chosen_optimizer": action}

        return self.state, reward, terminated, truncated, info

        
    








        

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env

feature_columns = [
    "initial_num_qubits",
    "initial_depth",
    "initial_two_qubit_gates",
    "initial_total_gates",
    "input_two_qubit_depth",
    "original_category_clifford",
    'original_category_feynman',
    'original_category_guoq_ibmnew',
    'original_category_misc',
    'original_category_qaoa', 
    'original_category_qft',
    'original_category_qv',
    'original_category_square-heisenberg'
]

train_env = OptimizerEnv(train_df, feature_columns)
test_env = OptimizerEnv(test_df, feature_columns)

check_env(train_env)

In [ ]:
model = DQN(
    "MlpPolicy",
    train_env,
    learning_rate=1e-3,
    buffer_size=10_000,
    learning_starts=500,
    batch_size=64,
    gamma=1.0,  # one-step episode, so discount barely matters
    verbose=1,
)

model.learn(total_timesteps=20_000)

model.save("optimizer_dqn")

In [ ]:
train_rewards = []
selected_optimizers = []

for circuit in train_df["original_circuit"].unique():

    circuit_rows = train_df[
        train_df["original_circuit"] == circuit
    ]

    # Use first row to get features
    row = circuit_rows.iloc[0]

    obs = row[feature_columns].to_numpy(dtype=np.float32)

    action, _ = model.predict(obs, deterministic=True)
    action = int(action)

    optimizer = train_env._action_to_optimizer[action]

    chosen_row = circuit_rows[
        circuit_rows["opt_1"] == optimizer
    ].iloc[0]

    reward = (
        chosen_row["initial_two_qubit_gates"]
        - chosen_row["final_two_qubit_gates"]
    ) / chosen_row["initial_two_qubit_gates"]

    train_rewards.append(reward)
    selected_optimizers.append(optimizer)

print("Train reward:", np.mean(train_rewards))
Counter(selected_optimizers)

In [ ]:


test_rewards = []
selected_optimizers = []

for circuit in test_df["original_circuit"].unique():

    circuit_rows = test_df[
        test_df["original_circuit"] == circuit
    ]

    # choose one row just to get features
    test_env.current_row = circuit_rows.iloc[0]
    obs = (
        test_env.current_row[feature_columns]
        .to_numpy(dtype=np.float32)
    )

    action, _ = model.predict(obs, deterministic=True)
    action = int(action)

    optimizer = test_env._action_to_optimizer[action]

    chosen_row = circuit_rows[
        circuit_rows["opt_1"] == optimizer
    ].iloc[0]

    reward = (
        chosen_row["initial_two_qubit_gates"]
        - chosen_row["final_two_qubit_gates"]
    ) / chosen_row["initial_two_qubit_gates"]

    test_rewards.append(reward)
    selected_optimizers.append(optimizer)

print(np.mean(test_rewards))

In [ ]:
from collections import Counter
Counter(selected_optimizers)

In [ ]:
import numpy as np

optimizer_names = [
    "qiskit_standard",
    "qiskit_ai",
    "tket",
    "wisq_rules",
    "wisq_bqskit"
]

random_rewards = []

for circuit in test_df["original_circuit"].unique():

    circuit_rows = test_df[
        test_df["original_circuit"] == circuit
    ]

    chosen_optimizer = np.random.choice(optimizer_names)

    chosen_row = circuit_rows[
        circuit_rows["opt_1"] == chosen_optimizer
    ].iloc[0]

    reward = (
        chosen_row["initial_two_qubit_gates"]
        - chosen_row["final_two_qubit_gates"]
    ) / chosen_row["initial_two_qubit_gates"]

    random_rewards.append(reward)

print("Random baseline:")
print(f"Mean reward = {np.mean(random_rewards):.4f}")

In [ ]:
wisq_rewards = []

for circuit in test_df["original_circuit"].unique():

    circuit_rows = test_df[
        test_df["original_circuit"] == circuit
    ]

    chosen_row = circuit_rows[
        circuit_rows["opt_1"] == "wisq_bqskit"
    ].iloc[0]

    reward = (
        chosen_row["initial_two_qubit_gates"]
        - chosen_row["final_two_qubit_gates"]
    ) / chosen_row["initial_two_qubit_gates"]

    wisq_rewards.append(reward)

print("Always WISQ_BQSKIT baseline:")
print(f"Mean reward = {np.mean(wisq_rewards):.6f}")

In [ ]:
def oracle_reward(df):
    rewards = []

    for circuit in df["original_circuit"].unique():
        rows = df[df["original_circuit"] == circuit]

        r = (
            (rows["initial_two_qubit_gates"]
             - rows["final_two_qubit_gates"])
            / rows["initial_two_qubit_gates"]
        )

        rewards.append(r.max())

    return np.mean(rewards)

print("Train oracle:", oracle_reward(train_df))
print("Test oracle:", oracle_reward(test_df))